In [2]:
import importlib
import download
importlib.reload(download)

<module 'download' from '/home/ravi/work/mining/project/download/__init__.py'>

In [3]:
import pandas as pd

In [ ]:
from search import search_news

urls = search_news(8000)
df = pd.DataFrame(urls, columns=['url'])
df.head()

In [19]:
from download import fetch_and_write_all

file_paths = await fetch_and_write_all(urls)
df['path'] = file_paths
df.head()

,url,path
0,http://www.newswire.ca/en/releases/archive/Feb...,articles/February202623c1544.html
1,http://www.newswire.ca/en/releases/archive/Feb...,articles/February202623c5893.html
2,http://www.newswire.ca/en/releases/archive/Feb...,articles/February202623c4520.html
3,http://www.newswire.ca/en/releases/archive/Feb...,articles/February202623c4355.html
4,http://www.newswire.ca/en/releases/archive/Feb...,articles/February202623c1563.html


In [21]:
df.index.name = 'id'
df.to_csv("data.csv")

In [4]:
df = pd.read_csv("data.csv")
df.set_index('id', inplace=True)

In [ ]:
import html_processor
import project_name_tagger
importlib.reload(html_processor)
importlib.reload(project_name_tagger)

from html_processor import get_article
from project_name_tagger import get_project_names
from tqdm.notebook import tqdm
import time

def p_name_helper(title, provider, published_date, body):
    try:
        project_names = get_project_names(f"TITLE: {title}, PROVIDER: {provider}, PUBLISHED_DATE: {published_date}, BODY: {body}")
        return str(project_names)
    except:
        return None

times = 0
total = len(df)
processed_count = 0

if 'project_names' not in df.columns:
    df['project_names'] = ''

for i, [idx, row] in tqdm(enumerate(df.iterrows()), total=total):
    start = time.time()

    if i < 1900:
        continue

    existing_projects = df.at[idx, 'project_names']
    if existing_projects is not None and pd.notna(existing_projects) and existing_projects != '[]' and existing_projects != '':
        # print('exists')
        continue
        
    try:
        with open(row.get('path'), 'r') as file:
            title, provider, published_date, body = get_article(file.read())
    except:
        # print('exception')
        continue

    if len(body) / 3 > 10000:
        # print('too long')
        continue

    result = p_name_helper(title, provider, published_date, body)
    df.at[idx, 'project_names'] = result
    
    if i%50 == 0:
        df.to_csv("data.csv")

    processed_count += 1
        
    end = time.time()
    times += end - start
    avg_time = times / processed_count

    # print(f"Processed {processed_count} docs, Average time per doc: {avg_time:.4f} seconds", end='\n')
    print(f"Processed {processed_count} docs, Average time per doc: {avg_time:.4f} seconds, estimated time: {(total - i) * avg_time / 60 / 60:.4f} hours", end='\r')

  0%|          | 0/8000 [00:00<?, ?it/s]

Processed 3756 docs, Average time per doc: 7.2110 seconds, estimated time: 4.4628 hourss

In [5]:
df

,url,path,project_names
id,,,
0,http://www.newswire.ca/en/releases/archive/Feb...,articles/February202623c1544.html,['Westgold Resources Limited (ASX: WGX)']
1,http://www.newswire.ca/en/releases/archive/Feb...,articles/February202623c5893.html,"['Berens Bridge & Road Project', 'PAK Lithium ..."
2,http://www.newswire.ca/en/releases/archive/Feb...,articles/February202623c4520.html,"['Columba High Grade Silver Project', 'Kootena..."
3,http://www.newswire.ca/en/releases/archive/Feb...,articles/February202623c4355.html,"['Big Bear Copper Property', 'Black Mammoth Me..."
4,http://www.newswire.ca/en/releases/archive/Feb...,articles/February202623c1563.html,"['El Quevar', 'Mani-Copan', 'Carmen']"
...,...,...,...
7995,http://www.newswire.ca/en/releases/archive/Jan...,articles/January202431c5309.html,NaN
7996,http://www.newswire.ca/en/releases/archive/Jan...,articles/January202431c5302.html,NaN
7997,http://www.newswire.ca/en/releases/archive/Jan...,articles/January202430c3204.html,NaN


In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   url            8000 non-null   str  
 1   path           7968 non-null   str  
 2   project_names  5589 non-null   str  
dtypes: str(3)
memory usage: 187.6 KB


In [7]:
df.describe()

,url,path,project_names
count,8000,7968,5589
unique,7999,7967,4814
top,http://www.newswire.ca/en/releases/archive/Jul...,articles/July202508c9439.html,"['Lunahuasi', 'Los Helados']"
freq,2,2,26
